# 00 — Download Data & Schema Audit

This notebook downloads the **akhauriyash/Code-Regression** dataset from HuggingFace and
performs a thorough schema audit to verify column types, partition integrity, and the
critical `val_accuracy` / `metric_type` overloading across the three dataset partitions
(APPS, CDSS, KBSS).

**What this notebook does:**
1. Downloads the Parquet files from HuggingFace.
2. Loads the full (or sampled) dataset.
3. Audits the schema — column names, dtypes, nulls.
4. Asserts partition-level metric-type integrity.
5. Inspects sample rows from each partition.
6. Summarises the `val_accuracy` distribution per partition.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

VALIDATION_MODE = os.environ.get("CODE_REGRESSION_FULL", "0") != "1"
print(f"\ud83d\udd2c VALIDATION_MODE = {VALIDATION_MODE}")
if VALIDATION_MODE:
    print("   Running with limited data (~500 rows/partition). Set CODE_REGRESSION_FULL=1 for full run.")
else:
    print("\ud83d\ude80 FULL MODE \u2014 using complete dataset.")

## 1. Download Dataset

In [ ]:
from utils.data_loader import download_dataset, load_full_dataset, audit_schema

# Download Parquet files from HuggingFace
downloaded_files = download_dataset()
print(f"\n\u2705 Downloaded {len(downloaded_files)} files:")
for f in downloaded_files:
    print(f"   {f}")

## 2. Schema Audit

We load the dataset and audit the schema to verify:
- Column names and data types are as expected.
- Partition counts across APPS, CDSS, and KBSS.
- The critical `val_accuracy` / `metric_type` overloading — `val_accuracy` stores
  **memory (bytes)** for APPS & CDSS but **latency (ms)** for KBSS, with the
  `metric_type` column disambiguating between them.

In [ ]:
import pandas as pd

df = load_full_dataset(validation_mode=VALIDATION_MODE)
print(f"Loaded {len(df):,} rows")
print()
audit_schema(df)

## 3. Partition Integrity Assertions

We verify that the `metric_type` column matches the expected values for each partition:
- **APPS** and **CDSS** → `metric_type == 'memory_bytes'`
- **KBSS** → `metric_type == 'latency_ms'`

These assertions guard against data corruption or unexpected schema changes upstream.

In [ ]:
# Verify metric_type integrity per partition
for space in ['APPS', 'CDSS', 'KBSS']:
    partition = df[df['space'] == space]
    if len(partition) == 0:
        print(f"\u26a0\ufe0f  No rows found for {space}")
        continue
    
    metric_types = partition['metric_type'].unique()
    print(f"\n{space} partition:")
    print(f"  Rows: {len(partition):,}")
    print(f"  metric_type values: {metric_types}")
    
    if space == 'KBSS':
        assert all(partition['metric_type'] == 'latency_ms'), \
            f"KBSS should have metric_type='latency_ms', got {metric_types}"
        print("  \u2705 Assertion passed: metric_type == 'latency_ms'")
    else:
        assert all(partition['metric_type'] == 'memory_bytes'), \
            f"{space} should have metric_type='memory_bytes', got {metric_types}"
        print(f"  \u2705 Assertion passed: metric_type == 'memory_bytes'")

print("\n" + "="*50)
print("All integrity assertions PASSED \u2705")

## 4. Sample Rows Inspection

Let's look at a few sample rows from each partition to understand the data format,
including the `input` text structure and `metadata` fields.

In [ ]:
for space in ['APPS', 'CDSS', 'KBSS']:
    partition = df[df['space'] == space]
    if len(partition) == 0:
        continue
    print(f"\n{'='*70}")
    print(f"Sample rows from {space} (showing 3 rows):")
    print(f"{'='*70}")
    sample = partition.head(3)
    for idx, row in sample.iterrows():
        print(f"\n--- Row {idx} ---")
        print(f"  identifier: {row['identifier']}")
        print(f"  space: {row['space']}")
        print(f"  metric_type: {row['metric_type']}")
        print(f"  val_accuracy: {row['val_accuracy']}")
        print(f"  target_metric: {row['target_metric']}")
        # Show first 200 chars of input
        input_preview = str(row['input'])[:200]
        print(f"  input (first 200 chars): {input_preview}...")
        # Show first 200 chars of metadata
        meta_preview = str(row['metadata'])[:200]
        print(f"  metadata (first 200 chars): {meta_preview}...")

## 5. val_accuracy Distribution Summary (per partition)

A quick look at the distribution of the target variable (`val_accuracy`) within each
partition. Remember that the semantics differ:
- **APPS / CDSS**: `val_accuracy` represents **memory usage in bytes**.
- **KBSS**: `val_accuracy` represents **latency in milliseconds**.

In [ ]:
print("val_accuracy summary statistics by partition:\n")
for space in ['APPS', 'CDSS', 'KBSS']:
    partition = df[df['space'] == space]
    if len(partition) == 0:
        continue
    print(f"--- {space} ({partition['metric_type'].iloc[0]}) ---")
    print(partition['val_accuracy'].describe().to_string())
    null_count = partition['val_accuracy'].isna().sum()
    print(f"  null count: {null_count}")
    print()

## Conclusion

**Key findings from this notebook:**

1. The dataset was successfully downloaded from HuggingFace (`akhauriyash/Code-Regression`).
2. The schema audit confirmed the expected columns and data types.
3. Partition integrity assertions passed — `metric_type` correctly reflects
   `memory_bytes` for APPS/CDSS and `latency_ms` for KBSS.
4. Sample row inspection shows the structure of `input` (code text) and `metadata` fields.
5. The `val_accuracy` distributions vary significantly across partitions, as expected
   given the different underlying metrics.

**Next →** Proceed to `01_eda.ipynb` for exploratory data analysis, including
distribution plots, correlation analysis, and outlier detection.